In [2]:
from argparse import ArgumentParser

from spacegraph_codebase.utils import *
from spacegraph_codebase.Place2Vec.cur_data_utils import load_pointset
from spacegraph_codebase.data_utils import load_ng
from spacegraph_codebase.model import NeighGraphEncoderDecoder
from spacegraph_codebase.train_helper import run_train, run_eval, run_joint_train
from spacegraph_codebase.trainer import *

from torch import optim
import numpy as np
import shlex
from spacegraph_codebase.trainer import make_args_parser, make_args_combine
parser = make_args_parser()
from pathlib import Path

In [3]:
# sh_file = exp_dir / "scripts" / "Place2Vec_2_enc_dec_global_rbf.sh"
exp_dir = Path("experiments_1028")
sh_file = exp_dir / "scripts" / "wl_1028_db4_scale_500_stg_1_sigma_0.5_lambda_0.5_seed_1.sh"
# output_dir = Path(exp_dir) / "clustering"

with open(sh_file, "r") as f:
    command = f.read()

command = sh_file.read_text(encoding="utf-8")

# 行末の \ による改行を結合
command = command.replace("\\\n", " ")

# コメント・空行を除外し、単語に分割
tokens = shlex.split(command, comments=True)

# trainモジュールの位置を探す
module_name = "spacegraph_codebase.Place2Vec.train"

if module_name not in tokens:
    print(f"SKIP: {sh_file.name}")

start = tokens.index(module_name) + 1
args_list = tokens[start:]

args = parser.parse_args(args_list)


# モデル生成
# g_enc_dec = make_enc_dec(args)
# g_enc_dec.eval()


In [12]:
# load dataset
print("Loading NeighGraph data..")

print("Loading training NeighGraph data..")
train_ng_list = load_ng(args.data_dir + "/neighborgraphs_training.pkl")
print("Loading validation NeighGraph  data..")
val_ng_list = load_ng(args.data_dir + "/neighborgraphs_validation.pkl")
print("Loading testing NeighGraph data..")
test_ng_list = load_ng(args.data_dir + "/neighborgraphs_test.pkl")


Loading NeighGraph data..
Loading training NeighGraph data..
Loading validation NeighGraph  data..
Loading testing NeighGraph data..


In [16]:
train_ng_list[0].center_pt
train_ng_list[0].context_pts

(18884, 12284, 19332, 787, 2315, 9244, 12011, 11043, 20612, 1235)

In [17]:
train_ng_list[0].neg_samples

[np.int64(20101),
 np.int64(10197),
 np.int64(2733),
 np.int64(2367),
 np.int64(12198),
 np.int64(12030),
 np.int64(13495),
 np.int64(1079),
 np.int64(14693),
 np.int64(2941)]